Note: This notebook is only for your reference, you do not have to include this in the final uploaded files.

# Group Project: Autonomous Driving Perception Pipeline

**CS5493: Topics in Autonomous Driving**

For any questions regarding this project, please contact Mr. REN Tianchi by email `tc.ren@my.cityu.edu.hk`

---

## 1. Overview

This group project asks each team to build, evaluate, and analyze a **complete perception pipeline** for autonomous driving using the nuScenes dataset. Teams of **three students are recommended**. The pipeline covers three sequential stages:

1. **3D Object Detection**: Detect objects from LiDAR point clouds using classical algorithms or deep learning models.
2. **Multi-Object Tracking**: Associate detections across frames to maintain consistent object identities using Kalman Filter and Hungarian algorithm.
3. **Trajectory Prediction**: Forecast future positions of tracked objects based on their observed trajectories.

### Learning Objectives

- Understand the **coordinate conventions and output formats** used by the nuScenes perception pipeline.
- Implement classical point cloud processing algorithms (**voxelization, RANSAC, DBSCAN**) and build a rule-based 3D object detector.
- Implement a **multi-object tracker** using a Kalman Filter for state estimation and the Hungarian algorithm for data association.
- Implement **trajectory prediction** methods and analyze how upstream errors propagate through the pipeline.
- Perform evaluation using the provided grading interface.

---
## 2. Dataset and Provided Resources

### 2.1 nuScenes Dataset

The nuScenes dataset is a large-scale autonomous driving dataset containing 1000 driving scenes. Each scene is 20 seconds long and annotated with 3D bounding boxes at 2 Hz. For this project, we use **nuScenes v1.0-mini** (10 scenes), consistent with the previous assignments. If you are interested and have sufficient compute resources, you may additionally experiment with the full dataset.

![nuScenes Dataset Structure](img/nuscenesdatasetstructure.png)

*Figure 1: Composition of the nuScenes dataset. The figure highlights the relationship between keyframe sensor samples, intermediate sweeps, and the metadata used throughout this project.*

![Track Prediction Example](img/track_prediction.gif)

*Figure 2: Example frame from a nuScenes temporal sequence. The LiDAR point cloud and 3D bounding boxes illustrate the sequential perception setting that motivates detection, tracking, and trajectory prediction.*

### 2.2 Provided Materials

| File / Directory | Description |
|---|---|
| `provided_baselines/` | Precomputed baseline outputs and reference metrics |
| `grading/` | **Grading script** — evaluates all three stages of the pipeline |
| `grading/scene_splits.json` | Per-scene observation/prediction frame splits |

---
## 3. Environment Setup

Run the following cells to set up the environment. Make sure you have created a conda environment first:

```bash
# Create conda environment
conda create -n nuscenes-project python=3.10 -y
conda activate nuscenes-project

# Install dependencies
pip install -r requirements.txt

# Verify setup
./run_smoke.sh
```

Key packages: `nuscenes-devkit`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`, `pandas`.

In [ ]:
# Import core packages
import numpy as np
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from pyquaternion import Quaternion

In [ ]:
# Load nuScenes mini dataset
import os
import sys

# Ensure the project root is on the path
PROJECT_ROOT = os.path.dirname(os.path.abspath('.'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, '.')

from nuscenes.nuscenes import NuScenes

DATAROOT = 'v1.0-mini'  # <-- adjust to your nuScenes data root
nusc = NuScenes(version='v1.0-mini', dataroot=DATAROOT, verbose=True)
print(f"Loaded {len(nusc.scene)} scenes, {len(nusc.sample)} samples")

---
## 4. Grading Interface

Understanding the grading interface is essential for producing a correct submission.

> **Important:** The grading script evaluates the three stages sequentially:
>
> 1. **Detection**: Evaluates `detection_results.json` on observation frames using official nuScenes detection metrics.
> 2. **Tracking**: Evaluates `tracking_results.json` on observation frames using MOT metrics (MOTA, MOTP, ID Switches, Fragmentation).
> 3. **Prediction**: The script identifies ground truth objects that appear in both the observation and prediction portions, matches them to student tracks, extracts the student's track histories, calls `predict_trajectory()`, and evaluates predictions against ground truth positions.

In [ ]:
# Demo: Initialize the grading interface
from grading.evaluate import ProjectGrader

grader = ProjectGrader(nusc, observation_ratio=0.75)

# Get observation sample tokens (for detection/tracking)
obs_tokens = grader.get_observation_sample_tokens()
pred_tokens = grader.get_prediction_sample_tokens()

print(f"Total observation tokens: {len(obs_tokens)}")
print(f"Total prediction tokens:  {len(pred_tokens)}")
print(f"First 5 observation tokens:")
for t in obs_tokens[:5]:
    print(f"--{t}")

---
## 5. Evaluation Protocol

For each scene in the dataset, frames are split into two portions:

| Portion | Purpose |
|---|---|
| Observation (first 75% of frames) | Detection and tracking evaluation |
| Prediction (last 25% of frames) | Trajectory prediction evaluation |

We provide a **grading script** (`grading/evaluate.py`) for evaluating the full pipeline. Students should **not** implement their own scoring for final reporting; the provided grading script is the authoritative evaluator.

### Prediction Interface

Students must implement the following function:



In [ ]:
def predict_trajectory(track_history, num_future_steps):
    """
    Predict future trajectory of an object.

    Args:
        track_history: list of dicts, sorted by timestamp, each with:
            - 'x': float (global x position)
            - 'y': float (global y position)
            - 'vx': float (x velocity)
            - 'vy': float (y velocity)
            - 'timestamp': int (microseconds)
        num_future_steps: int, number of future positions to predict

    Returns:
        list of (x, y) tuples, one per future step
    """

The `track_history` is extracted from the **student's own tracking results** rather than from ground truth. This means:
- If the student's tracker produces noisy position estimates, the prediction input will be noisy.
- If the tracker loses an object early, the track history will be shorter.
- This reflects real-world conditions where perception modules are chained.

---
## 6. Project Tasks

This project is divided into four phases. Each phase builds on the previous one.

### Question 1 (Coordinate Frames)

> The project code processes LiDAR detections in local sensor coordinates, but the submission files must report object states in the *global* frame. Explain the roles of the *sensor*, *ego vehicle*, and *global* coordinate systems in nuScenes, and describe how a detection should be transformed from the LiDAR frame into the global frame before writing JSON results.
>
> **Answer in your report.**

### Question 2 (Point Cloud Density)

> How many LiDAR points does a typical nuScenes sample contain? Compute the average and standard deviation across all samples in the mini split. What implications does the point density have for voxel-based vs. point-based detection methods?
>
> **Answer in your report.**

In [ ]:
# Demo: Visualize a LiDAR point cloud (Bird's Eye View)
sample_token = obs_tokens[0]
points = get_lidar_points_in_global(nusc, sample_token)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.scatter(points[:, 0], points[:, 1], s=0.2, c=points[:, 2], cmap='viridis', alpha=0.5)
ax.set_xlabel('X (global)')
ax.set_ylabel('Y (global)')
ax.set_title(f'LiDAR BEV — {points.shape[0]} points')
ax.set_aspect('equal')
plt.colorbar(ax.collections[0], ax=ax, label='Z height (m)')
plt.tight_layout()
plt.show()

---
### Phase 1: 3D Object Detection

In this phase, you will implement a rule-based 3D object detector using classical algorithms. Run detection on the **observation portion** (the first 75% of frames) of each scene.

Use `grader.get_observation_sample_tokens()` to determine which frames to process.

**Pipeline Steps:**

1. **Voxelization**: Implement `voxelize(points, voxel_size, pc_range)` — partition the point cloud into a 3D voxel grid and return non-empty voxel centers and feature means.
2. **RANSAC ground segmentation**: Implement `ransac_ground_removal(points, ...)` — fit a ground plane and remove ground points.
3. **DBSCAN clustering**: Implement `dbscan_cluster(points, eps, min_samples)` — cluster non-ground points into potential objects.
4. **Heuristic classification & bounding box fitting**: Classify each cluster by geometry (e.g., length > 4m → car) and fit a 3D bounding box.
5. **nuScenes format output**: Save detections to `submissions/detection_results.json`.

#### Step 1: Voxelization

Implement the function in `src/detection/voxelization.py`:

In [ ]:
from src.detection.voxelization import voxelize

#### Step 2: RANSAC Ground Segmentation

Implement the function in `src/detection/ransac.py`:

In [ ]:
from src.detection.ransac import ransac_ground_removal

#### Step 3: DBSCAN Clustering

Implement the function in `src/detection/dbscan_cluster.py`:

In [ ]:
from src.detection.dbscan_cluster import dbscan_cluster

#### Step 4: Bounding Box Fitting & Heuristic Classification

Implement the functions in `src/detection/bbox_fitting.py`:

**Classification guidelines (metres):**

| Class | Length | Width | Height |
|---|---|---|---|
| car | 3.5–6.0 | 1.5–2.5 | — |
| truck | 6.0–12.0 | — | — |
| bus | >10 | — | — |
| pedestrian | <1.0 | — | 1.2–2.0 |
| bicycle/motorcycle | 1.2–2.5 | — | <1.8 |
| traffic_cone | very small | — | <1.0 |
| barrier | long & thin | — | <1.2 |

In [ ]:
from src.detection.bbox_fitting import fit_bounding_box, classify_cluster

print("nuScenes detection classes:")
from src.utils import DETECTION_NAMES
for i, name in enumerate(DETECTION_NAMES):
    print(f"{i+1}. {name}")

#### Step 5: Complete Detection Pipeline

Implement `detect_single_sample()` in `src/detection/detector.py` to chain all steps together:

In [ ]:
from src.detection.detector import LidarDetector

print("LidarDetector default configuration:")
for key, val in LidarDetector.DEFAULT_CONFIG.items():
    print(f"  {key}: {val}")

### Question 3 (Architecture)

> Explain the difference between voxel-based (e.g., SECOND) and pillar-based (e.g., PointPillars) point cloud representations. What trade-offs does each approach make in terms of spatial resolution and computational cost? Relate this to your own voxelization implementation.
>
> **Answer in your report.**

---
### Phase 2: Multi-Object Tracking

In this phase, you will implement a multi-object tracker that associates detections across frames. The tracker may be a simple Kalman Filter-based tracker (as in Assignment 2), but it must be implemented from scratch. You may also implement a more advanced tracker (e.g., SORT or DeepSORT) if you wish.

The tracker runs on the observation frames, **using your detection results from Phase 1 as input**.

**Key components to implement:**
- Kalman Filter (`src/tracking/kalman_filter.py`)
- Multi-Object Tracker (`src/tracking/tracker.py`)

#### Kalman Filter

Implement in `src/tracking/kalman_filter.py`:

The filter maintains:
- **State** $\mathbf{x} = [p_x, p_y, p_z, v_x, v_y, v_z]^\top$ (6D)
- **Covariance** $\mathbf{P}$ (6×6)

**Motion model** (discrete, dt seconds):
$$\mathbf{x}_{k+1} = \mathbf{F} \cdot \mathbf{x}_k + \mathbf{w}, \quad \mathbf{w} \sim \mathcal{N}(0, \mathbf{Q})$$

**Measurement model**:
$$\mathbf{z}_k = \mathbf{H} \cdot \mathbf{x}_k + \mathbf{v}, \quad \mathbf{v} \sim \mathcal{N}(0, \mathbf{R})$$

**Predict step**:
$$\mathbf{x}^- = \mathbf{F} \cdot \mathbf{x}, \quad \mathbf{P}^- = \mathbf{F} \cdot \mathbf{P} \cdot \mathbf{F}^\top + \mathbf{Q}$$

**Update step**:
$$\mathbf{y} = \mathbf{z} - \mathbf{H} \cdot \mathbf{x}^-$$
$$\mathbf{S} = \mathbf{H} \cdot \mathbf{P}^- \cdot \mathbf{H}^\top + \mathbf{R}$$
$$\mathbf{K} = \mathbf{P}^- \cdot \mathbf{H}^\top \cdot \mathbf{S}^{-1}$$
$$\mathbf{x} = \mathbf{x}^- + \mathbf{K} \cdot \mathbf{y}, \quad \mathbf{P} = (\mathbf{I} - \mathbf{K} \cdot \mathbf{H}) \cdot \mathbf{P}^-$$

In [ ]:
# Demo: Kalman Filter matrices setup
# This shows what your __init__ should construct

dt = 0.5  # nuScenes keyframe rate = 2 Hz

# State transition matrix F (constant velocity model)
F = np.eye(6)
F[0, 3] = dt  # px += vx * dt
F[1, 4] = dt  # py += vy * dt
F[2, 5] = dt  # pz += vz * dt

# Measurement matrix H (we observe position only)
H = np.zeros((3, 6))
H[0, 0] = 1  # observe px
H[1, 1] = 1  # observe py
H[2, 2] = 1  # observe pz

#### Multi-Object Tracker

Implement `MultiObjectTracker.update()` in `src/tracking/tracker.py`:

**Steps per frame:**
1. Predict all existing tracks forward.
2. Build cost matrix (centre distance between predicted track positions and new detections).
3. Solve assignment with **Hungarian algorithm** (`scipy.optimize.linear_sum_assignment`).
4. Update matched tracks; create new tracks for unmatched detections; age unmatched tracks.
5. Prune dead tracks (tracks exceeding `max_age` without updates).
6. Return nuScenes tracking entries for confirmed tracks.

### Question 4 (Tracking)

> Explain the roles of the Kalman Filter `predict()` and `update()` steps in the tracking pipeline. How does the Hungarian algorithm handle the case where the number of detections differs from the number of existing tracks? Describe what happens to unmatched detections and unmatched tracks.
>
> **Answer in your report.**

---
### Phase 3: Trajectory Prediction

In this phase, you will implement trajectory prediction. We provide a baseline constant-velocity (CV) prediction. You must then implement an improved method.

The grading script constructs prediction targets by matching your tracking results to ground truth objects and extracting the corresponding track histories **from your tracker's output**.

**Constant Velocity (CV) baseline**:

$$x_{t+k} = x_t + v_x \cdot k \cdot \Delta t, \quad y_{t+k} = y_t + v_y \cdot k \cdot \Delta t$$

where $\Delta t = 0.5$s (2 Hz sampling rate) and $(v_x, v_y)$ are estimated from recent history.

**Improved prediction methods** (pick ≥ 1):
- Kalman Filter-based prediction (reuse or extend the filter from Phase 2)
- Polynomial / spline trajectory fitting
- Acceleration-aware models
- Category-aware prediction (different models for vehicles vs. pedestrians)

In [ ]:
# Demo: Constant Velocity prediction baseline (already provided in src/prediction/predictor.py)

from src.prediction.predictor import predict_cv

# Simulate a track history
example_history = [
    {'x': 100.0, 'y': 200.0, 'vx': 5.0, 'vy': 2.0, 'timestamp': 1000000},
    {'x': 102.5, 'y': 201.0, 'vx': 5.0, 'vy': 2.0, 'timestamp': 1500000},
    {'x': 105.0, 'y': 202.0, 'vx': 5.0, 'vy': 2.0, 'timestamp': 2000000},
]

predictions = predict_cv(example_history, num_future_steps=5)

print("Track history:")
for h in example_history:
    print(f"--t={h['timestamp']}: ({h['x']:.1f}, {h['y']:.1f}), v=({h['vx']:.1f}, {h['vy']:.1f})")

print(f"Predicted future positions (dt=0.5s):")
for i, (px, py) in enumerate(predictions):
    print(f"--Step {i+1}: ({px:.1f}, {py:.1f})")

In [ ]:
# Demo: Visualize predicted trajectory

fig, ax = plt.subplots(figsize=(10, 6))

# Plot history
hist_x = [h['x'] for h in example_history]
hist_y = [h['y'] for h in example_history]
ax.plot(hist_x, hist_y, 'bo-', markersize=8, label='Track History', linewidth=2)

# Plot predictions
pred_x = [p[0] for p in predictions]
pred_y = [p[1] for p in predictions]
ax.plot(pred_x, pred_y, 'r^--', markersize=8, label='CV Prediction', linewidth=2)

# Connect last observed to first predicted
ax.plot([hist_x[-1], pred_x[0]], [hist_y[-1], pred_y[0]], 'k--', alpha=0.5)

ax.set_xlabel('X (global)')
ax.set_ylabel('Y (global)')
ax.set_title('Constant Velocity Prediction Demo')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Question 5 (Prediction)

> How does prediction error grow as the prediction horizon increases? Compare the error growth pattern of your CV baseline versus your improved method. What factors cause the CV model to degrade over longer horizons, and how does your improved method address them?
>
> **Answer in your report (with supporting figures).**

---
### Phase 4: Report & Submission

1. Write a project report (maximum 6 pages, in PDF or DOCX format) covering:
   - Introduction and problem setup
   - Coordinate conventions and how detections/tracks are represented in the global frame
   - Detection: algorithm description and design choices
   - Tracking: Kalman Filter design, data association strategy, track management
   - Prediction: baseline and improved methods, comparison analysis
   - Evaluation results for all three stages
   - Answers to all 5 analysis questions
   - Division of work among team members

2. Prepare the final code repository with a clear `README.md`.

3. Run the grading script (`grading/evaluate.py`) and include the output in your submission.

---
## 7. Evaluation Metrics

### 7.1 Detection Metrics

| Metric | Description |
|---|---|
| mAP | Mean Average Precision over all 10 detection classes |
| NDS | nuScenes Detection Score: weighted combination of mAP and TP errors |
| mATE | Mean Average Translation Error (meters) |
| mASE | Mean Average Scale Error (1 − IoU after alignment) |
| mAOE | Mean Average Orientation Error (radians) |
| mAVE | Mean Average Velocity Error (m/s) |
| mAAE | Mean Average Attribute Error (classification accuracy) |

### 7.2 Tracking Metrics

| Metric | Description |
|---|---|
| MOTA | Multiple Object Tracking Accuracy: $1 - (\text{FN} + \text{FP} + \text{IDSW}) / \text{GT}$ |
| MOTP | Multiple Object Tracking Precision: average distance of matched pairs |
| ID Switches | Number of identity changes for tracked objects |
| Fragmentation | Number of track breaks (gaps in frame continuity) |

### 7.3 Prediction Metrics

| Metric | Description |
|---|---|
| L1 Error (Mean) | Average Manhattan distance between predicted and ground truth positions |
| L2 Error (Mean) | Average Euclidean distance between predicted and ground truth positions |
| L1 Error (Median) | Median Manhattan distance (robust to outliers) |
| L2 Error (Median) | Median Euclidean distance (robust to outliers) |

---
## 8. Grading Rubric

The project is graded out of 100 points.

| Criterion | Points |
|---|---|
| Environment Setup | 20 |
| 3D Object Detection | 20 |
| Multi-Object Tracking | 20 |
| Trajectory Prediction | 15 |
| Report Quality | 15 |
| Code Reproducibility & Team Collaboration | 10 |
| **Total** | **100** |

**Important Note:** Please make sure to follow the submission requirements and use the provided grading script for evaluation. Points will be deducted for missing or incorrectly formatted outputs, failure to implement the required prediction function, and for any discrepancies between the report and the implemented code.

### Alternatives

If computing resources are available, you may adopt deep learning based tricks rather than rule-based methodology mentioned above, our grading will be based on model architecture, performance and the quality of the report if so.

---
## 9. Submission Formats

### 9.1 Detection Results (`submissions/detection_results.json`)

Standard nuScenes detection format. **Only include observation-portion sample tokens.**

```json
{
    "meta": {"use_camera": false, "use_lidar": true, ...},
    "results": {
        "<sample_token>": [
            {
                "sample_token": "<sample_token>",
                "translation": [x, y, z],
                "size": [w, l, h],
                "rotation": [qw, qx, qy, qz],
                "detection_name": "car",
                "detection_score": 0.85
            }
        ]
    }
}
```

### 9.2 Tracking Results (`submissions/tracking_results.json`)

Standard nuScenes tracking format. **Only include observation-portion sample tokens.**

```json
{
    "meta": {"use_camera": false, "use_lidar": true, ...},
    "results": {
        "<sample_token>": [
            {
                "sample_token": "<sample_token>",
                "translation": [x, y, z],
                "size": [w, l, h],
                "rotation": [qw, qx, qy, qz],
                "velocity": [vx, vy],
                "tracking_id": "track_001",
                "tracking_name": "car",
                "tracking_score": 0.85
            }
        ]
    }
}
```

### 9.3 Prediction Function (`predict_trajectory`)

Students must implement this function and ensure it is callable by the grading script. See Section 5 for the full signature.

---
## 10. Submission Requirements

Please submit the following items:

1. **Report**: A PDF or DOCX report (maximum 6 pages) covering the required analysis, implementation details, results, and team contribution breakdown.
2. **Code repository**: A clean, runnable repository with a clear `README.md`, dependency instructions, and the required implementation files.
3. **Detection output**: `submissions/detection_results.json` generated on the observation portion only.
4. **Tracking output**: `submissions/tracking_results.json` generated on the observation portion only.
5. **Prediction function**: A callable implementation of `predict_trajectory(track_history, num_future_steps)` in the expected project code location.
6. **Evaluation evidence**: The output produced by running `grading/evaluate.py` on your final pipeline.

---
## Useful References

1. Caesar, H. et al., "nuScenes: A multimodal dataset for autonomous driving," CVPR 2020.  
   https://www.nuscenes.org/nuscenes

2. nuScenes Detection Challenge: https://www.nuscenes.org/object-detection

3. nuScenes Tracking Challenge: https://www.nuscenes.org/tracking

4. nuScenes Prediction Challenge: https://www.nuscenes.org/prediction

5. nuScenes DevKit GitHub: https://github.com/nutonomy/nuscenes-devkit